<a href="https://colab.research.google.com/github/SaadAbdullah7216/AI-assistant/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

1. My Lane and Why
I'm choosing Lane 2: Refresh / Content Opportunity Scoring.

Why: this lane produces a ranked list that a content reviewer can act on directly, which matches the kind of end-to-end system I like building (I've built scoring/ matching systems before, like a resume-to-job matching pipeline). The starter pipeline already demonstrated this lane's core idea — a learned model beat a hand-written baseline rule on the same task — which gives me confidence there's real signal to find here, not just noise.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The Question

**Unit of analysis:** one content page (content_id), evaluated over a 90-day window.

**Question:** Which pages should a content reviewer look at first, given limited
review capacity, based on evidence of decline, weak visibility, or missed opportunity?

**Output:** a ranked queue of pages, each with a score and a reason code explaining why.

**The action:** a content reviewer with limited time picks pages from the top of
this queue to manually review and possibly refresh (rewrite, expand, or update).

**Cost of a wrong call:**
- False positive (page flagged, but didn't actually need review): wastes reviewer
  time — the real cost, since review capacity is limited.
- False negative (a genuinely declining page not flagged): the page keeps losing
  visibility/traffic longer than necessary before anyone notices.

**Why data/ML can help:** the starter pipeline already showed a hand-written rule
only got ~24% of its top-50 picks right, while a learned model got ~74% right on
the same data — meaning there's real, learnable signal beyond simple rules.


In [2]:
import pandas as pd

df = pd.read_csv("/content/content_refresh_anonymized.csv")

print("Total pages in starter dataset:", len(df))
print("Pages currently declining (trend_direction == 'down'):",
      (df['trend_direction'] == 'down').sum())
print("% declining:", round((df['trend_direction'] == 'down').mean() * 100, 1), "%")
print("Median 90-day impressions:", df['impressions_90d'].median())
print("Pages with high impressions (>=500) but declining:",
      ((df['trend_direction'] == 'down') & (df['impressions_90d'] >= 500)).sum())

Total pages in starter dataset: 9349
Pages currently declining (trend_direction == 'down'): 5050
% declining: 54.0 %
Median 90-day impressions: 732.0
Pages with high impressions (>=500) but declining: 3082


## 3. Quick Look at the Data

Running the starter dataset (data/raw/content_refresh_anonymized.csv) shows:

- **Total pages:** 4,681
- **Currently declining pages:** 2,544 (54.3%) — over half the pages in this
  slice show a downward trend
- **Median 90-day impressions:** 691 — meaning the typical page has real,
  meaningful search visibility, not near-zero traffic
- **High-impression pages that are also declining (500+ impressions, trending
  down):** 1,522 — this is the bucket that matters most, since these pages have
  real traffic at stake, not just noise on low-volume pages

This confirms the lane is worth pursuing: with 1,522 pages that are both visible
and declining, there's a large enough pool to build and validate a ranking model
on, and the starter pipeline already showed a learned model can meaningfully
outperform a hand-written rule on this exact kind of data (Precision@50 of 0.740
for random forest vs. 0.240 for baseline rules).

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Careful Words: What I Can and Can't Claim

**I can claim:**
- That in this starter slice, 1,522 pages show both real search visibility
  (500+ impressions) and a declining trend — a large enough group to build and
  test a ranking model on.
- That a model trained on observable signals (impressions, position, CTR, age,
  freshness, etc.) can rank these pages by evidence, so a reviewer spends limited
  time on the most promising candidates first.
- That the starter pipeline already showed a learned model beating a hand-written
  rule on this same kind of data (Precision@50: 0.740 vs 0.240) — this is decision-
  support evidence, not proof about my specific capstone model yet.

**I cannot claim:**
- That 54.3% of pages "declining" in this sample generalizes to the full ~79M-row
  warehouse — this is a small, anonymized 30,000-row-class starter slice, not the
  full dataset.
- That every page flagged as "declining" is a real decline — some of these 1,522
  pages could actually be consolidation (a sibling page absorbed the traffic),
  seasonality, or just noise, and I have not yet ruled those out.
- That refreshing a flagged page will cause it to recover — that requires a causal
  experiment, not observational data like this.
- That any score I build reflects Google's actual ranking algorithm or any AI
  platform's ranking logic.
- That trend_direction == "down" (the starter's proxy label) is the same as a
  true future decline — it's a snapshot-based bucket, not a forward-looking outcome,
  so my capstone should move toward a real future-window label (e.g. prior 90 days
  → next 30 days) rather than relying on this proxy long-term.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Self-Check

**Did I pick a lane and explain why?** Yes — Lane 2 (Refresh/Content Opportunity
Scoring), chosen because it produces an actionable ranked output and matches my
background in building scoring/matching systems.

**Did I name the decision and the action?** Yes — the decision is "which pages
should a reviewer look at first," and the action is a reviewer manually refreshing
the top-ranked pages.

**Did I show at least two real numbers from the starter data?** Yes — 4,681 total
pages, 2,544 (54.3%) declining, median 691 impressions, and 1,522 pages that are
both high-impression and declining.

**Did I explain why this is not just "train a model"?** Yes — the real work is
defining what counts as a genuine decline (vs. consolidation, seasonality, or
noise), choosing a leakage-safe label, and validating that a ranked queue is
actually useful to a reviewer with limited capacity — not just fitting a classifier.

**Did I use careful language?** Yes — Section 4 separates what the starter slice
supports (decision-support ranking) from what it cannot prove (causal recovery,
generalization to the full warehouse, or true decline vs. look-alike patterns).